# Lab 3 | AI Constitutional Research Assistant
**Build a question-to-source search tool using authentic constitutional text.**

**Corpus:** selected U.S. constitutional amendments, transcribed by the [National Archives](https://www.archives.gov/founding-docs/bill-of-rights-transcript) and [Amendments 11–27](https://www.archives.gov/founding-docs/amendments-11-27).

**Goal:** use TF-IDF and cosine similarity to rank relevant passages; print the real text and a citation. The assistant does **not** decide cases, apply precedent, or give legal advice.

**Instructions:** Runtime → Run all; then edit `QUESTION` to research a new issue.

*Editorial search keywords are added to improve matching; they are not part of the Constitution.*

### 1 | Prepare a small legal source collection

In [ ]:
# Every entry contains an authentic constitutional passage and its reference.
# Search keywords are teaching aids, not quotations from the amendment.
records = [('Amendment I', 'Congress shall make no law respecting an establishment of religion, or prohibiting the free exercise thereof; or abridging the freedom of speech, or of the press; or the right of the people peaceably to assemble, and to petition the Government for a redress of grievances.', 'speaking opinions protest expression religious practice worship church news journalism newspapers press petition assembly free speech freedom of religion'), ('Amendment II', 'A well regulated Militia, being necessary to the security of a free State, the right of the people to keep and bear Arms, shall not be infringed.', 'weapons firearms guns arms militia'), ('Amendment III', 'No Soldier shall, in time of peace be quartered in any house, without the consent of the Owner, nor in time of war, but in a manner to be prescribed by law.', 'soldiers staying home housing military quartering owner'), ('Amendment IV', 'The right of the people to be secure in their persons, houses, papers, and effects, against unreasonable searches and seizures, shall not be violated, and no Warrants shall issue, but upon probable cause, supported by Oath or affirmation, and particularly describing the place to be searched, and the persons or things to be seized.', 'police searches home backpack car phone privacy warrant property search seizure probable cause'), ('Amendment V', 'No person shall be held to answer for a capital, or otherwise infamous crime, unless on a presentment or indictment of a Grand Jury, except in cases arising in the land or naval forces, or in the Militia, when in actual service in time of War or public danger; nor shall any person be subject for the same offence to be twice put in jeopardy of life or limb; nor shall be compelled in any criminal case to be a witness against himself, nor be deprived of life, liberty, or property, without due process of law; nor shall private property be taken for public use, without just compensation.', 'right to remain silent self incrimination forced confession double jeopardy due process government take property compensation'), ('Amendment VI', 'In all criminal prosecutions, the accused shall enjoy the right to a speedy and public trial, by an impartial jury of the State and district wherein the crime shall have been committed, which district shall have been previously ascertained by law, and to be informed of the nature and cause of the accusation; to be confronted with the witnesses against him; to have compulsory process for obtaining witnesses in his favor, and to have the Assistance of Counsel for his defence.', 'criminal charges lawyer attorney defense trial speedy jury witnesses court'), ('Amendment VII', 'In Suits at common law, where the value in controversy shall exceed twenty dollars, the right of trial by jury shall be preserved, and no fact tried by a jury, shall be otherwise re-examined in any Court of the United States, than according to the rules of the common law.', 'civil lawsuits money disputes jury civil case'), ('Amendment VIII', 'Excessive bail shall not be required, nor excessive fines imposed, nor cruel and unusual punishments inflicted.', 'jail bail punishment cruel excessive fines'), ('Amendment IX', 'The enumeration in the Constitution, of certain rights, shall not be construed to deny or disparage others retained by the people.', 'other rights not enumerated unlisted constitutional rights'), ('Amendment X', 'The powers not delegated to the United States by the Constitution, nor prohibited by it to the States, are reserved to the States respectively, or to the people.', 'states rights powers federal government reserved'), ('Amendment XIII, Section 1', 'Neither slavery nor involuntary servitude, except as a punishment for crime whereof the party shall have been duly convicted, shall exist within the United States, or any place subject to their jurisdiction.', 'slavery forced labor involuntary servitude'), ('Amendment XIV, Section 1', 'All persons born or naturalized in the United States, and subject to the jurisdiction thereof, are citizens of the United States and of the State wherein they reside. No State shall make or enforce any law which shall abridge the privileges or immunities of citizens of the United States; nor shall any State deprive any person of life, liberty, or property, without due process of law; nor deny to any person within its jurisdiction the equal protection of the laws.', 'citizenship born naturalized equal protection discrimination equality states due process'), ('Amendment XV, Section 1', 'The right of citizens of the United States to vote shall not be denied or abridged by the United States or by any State on account of race, color, or previous condition of servitude--', 'voting vote race race discrimination'), ('Amendment XIX', 'The right of citizens of the United States to vote shall not be denied or abridged by the United States or by any State on account of sex.', 'women voting vote sex'), ('Amendment XXVI, Section 1', 'The right of citizens of the United States, who are eighteen years of age or older, to vote shall not be denied or abridged by the United States or by any State on account of age.', 'voting age eighteen 18 young voters right vote')]
print(f"Source passages loaded: {len(records)}")
for ref, text, tags in records[:4]:
    print(" -", ref)

### 2 | Train a simple text-retrieval model

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# TF-IDF learns which words help distinguish one source from another.
# It is a traditional machine-learning text search technique, not a chatbot.
search_text = [text + " " + (tags + " ") * 2 for ref, text, tags in records]
vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
document_vectors = vectorizer.fit_transform(search_text)

def research(question, top_k=3):
    """Return ranked source excerpts; never invent a legal answer."""
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, document_vectors).ravel()
    print("QUESTION:", question)
    if scores.max() < 0.15:
        print("No strong match found. The source collection may not cover this issue.")
        return
    ranked = [i for i in scores.argsort()[::-1]
              if scores[i] >= max(0.15, 0.15 * scores.max())][:top_k]
    for rank, index in enumerate(ranked, start=1):
        reference, text, tags = records[index]
        print(f"\n{rank}. {reference} | similarity: {scores[index]:.2f}")
        print("SOURCE:", text)
    print("\nSource: U.S. National Archives | educational research only; not legal advice.")

### 3 | Ask a research question and inspect the cited sources

In [ ]:
# GROUP CHALLENGE: replace the question with your own scenario.
QUESTION = "What protects freedom of speech and peaceful protest?"
research(QUESTION)

### 4 | Test the assistant with several scenarios

In [ ]:
test_questions = [
    "Can the government restrict freedom of the press?",
    "What constitutional amendment discusses police searches and warrants?",
    "Does an accused person have a right to a lawyer in a criminal trial?",
    "At what age is the right to vote protected?",
    "Is my landlord required to fix a broken refrigerator?"  # Deliberately out of scope.
]
for question in test_questions:
    print("\n" + "=" * 70)
    research(question, top_k=1)

### Team mission
1. Ask the assistant **three different** constitutional questions. Record the top references.
2. Ask a question that the included sources **cannot answer** (for example, a private rental dispute).
3. Explain why a search similarity score is **not** the probability that a legal conclusion is correct.
4. Prepare a **30-second demonstration** showing a question, a retrieved amendment, and one limitation.

**Debrief:** How does retrieval from a trusted source differ from generating a confident-sounding legal answer?